In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.decomposition import PCA
import json

In [2]:
# =========================================================
# 1️⃣ LOAD DATA
# =========================================================
def load_data(input_path: str) -> pd.DataFrame:
    df = pd.read_parquet(input_path)
    print(f"[INFO] Loaded data: {df.shape}")
    return df

In [ ]:
FEATURE_PREFIX = 'feat_'

In [ ]:
# =========================================================
# 2️⃣ CATEGORICAL ENCODING
# =========================================================
def encode_categorical(df: pd.DataFrame) -> pd.DataFrame:
    object_cols = df.select_dtypes(include=["object"]).columns.tolist()
    low_cardinality_cols, medium_cardinality_cols, high_cardinality_cols = [], [], []

    # Split by cardinality
    for col in object_cols:
        n_unique = df[col].nunique()
        if n_unique < 10:
            low_cardinality_cols.append(col)
        elif 10 <= n_unique <= 100:
            medium_cardinality_cols.append(col)
        else:
            high_cardinality_cols.append(col)

    print(f"[INFO] Low: {len(low_cardinality_cols)}, Medium: {len(medium_cardinality_cols)}, High: {len(high_cardinality_cols)}")

    # One-hot encode low-cardinality
    if low_cardinality_cols:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        ohe_df = pd.DataFrame(
            ohe.fit_transform(df[low_cardinality_cols]),
            columns=ohe.get_feature_names_out(low_cardinality_cols),
            index=df.index
        )
    else:
        ohe_df = pd.DataFrame(index=df.index)

    # Frequency encode medium-cardinality
    freq_encoded = pd.DataFrame(index=df.index)
    for col in medium_cardinality_cols:
        freqs = df[col].value_counts(dropna=False)
        freq_encoded[col + "_freq"] = df[col].map(freqs)

    # Frequency encode high-cardinality
    high_card_freq_encoded = pd.DataFrame(index=df.index)
    for col in high_cardinality_cols:
        freqs = df[col].value_counts(dropna=False)
        high_card_freq_encoded[col + "_freq"] = df[col].map(freqs)

    # Combine all
    cat_encoded = pd.concat([ohe_df, freq_encoded, high_card_freq_encoded], axis=1)
    cat_encoded.columns = [FEATURE_PREFIX+each for each in cat_encoded.columns]
    
    # Drop original object cols and merge back
    df = pd.concat([df, cat_encoded], axis=1)
    print('Dropped column count in categorical encoding:', len(object_cols))
    print('No columns are DROPPED!!')
    print('Newly added column count in categorical encoding:', cat_encoded.shape[1])
    print(f"[INFO] After categorical encoding: {df.shape}")
    return df


In [ ]:
# =========================================================
# 3️⃣ NUMERIC TRANSFORMATIONS
# =========================================================
def transform_numerical(df: pd.DataFrame) -> pd.DataFrame:
    numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

    # Log transform
    if "TransactionAmt" in df.columns:
        df[FEATURE_PREFIX+"TransactionAmt_log"] = np.log1p(df["TransactionAmt"])

    # Standard scaling on C*, D* features
    std_cols = [c for c in df.columns if c.startswith(("C", "D")) and c in numerical_cols]
    if std_cols:
        scaler = StandardScaler()
        df[std_cols] = scaler.fit_transform(df[std_cols])

    # Robust scaling for heavy-tailed vars
    robust_cols = ["TransactionAmt_log"]
    if robust_cols:
        robust_scaler = RobustScaler()
        df[robust_cols] = robust_scaler.fit_transform(df[robust_cols])

    print(f"[INFO] After numeric transformations: {df.shape}")
    return df

In [5]:
# =========================================================
# 4️⃣ TEMPORAL FEATURES
# =========================================================
def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    if "TransactionDT" in df.columns:
        df["hour"] = (df["TransactionDT"] / 3600) % 24
        df["day"] = (df["TransactionDT"] / (3600 * 24)) % 7
        df["week"] = (df["TransactionDT"] / (3600 * 24 * 7))
        df["is_weekend"] = df["day"].isin([5, 6]).astype(int)
    
    print(f"[INFO] New columns added : hour, day, week, is_weekend")
    print(f"[INFO] After temporal features: {df.shape}")
    return df

In [6]:
# =========================================================
# 5️⃣ AGGREGATION FEATURES
# =========================================================
def add_aggregations(df: pd.DataFrame) -> pd.DataFrame:
    agg_configs = {
        "card1": ["mean", "std", "count"],
        "addr1": ["mean", "std"],
        "P_emaildomain": ["mean"]
    }
    newly_added_cols = 0
    for key, agg_funcs in agg_configs.items():
        if key in df.columns and "TransactionAmt" in df.columns:
            try:
                agg = df.groupby(key)["TransactionAmt"].agg(agg_funcs).reset_index()
                agg.fillna(0, inplace=True)
                agg.columns = [key] + [f"{key}_TransactionAmt_{f}" for f in agg_funcs]
                newly_added_cols += len(agg.columns) - 1  # Exclude key column
                df = df.merge(agg, on=key, how="left")
            except Exception as e:
                print(f"[WARN] Skipped {key} aggregation: {e}")
    print(f"[INFO] New columns added through aggregations ", newly_added_cols)
    print(f"[INFO] After aggregations: {df.shape}")
    return df



In [7]:
# =========================================================
# 6️⃣ RATIO & INTERACTION FEATURES
# =========================================================
def add_ratios_interactions(df: pd.DataFrame) -> pd.DataFrame:
    if "TransactionAmt" in df.columns and "card1_TransactionAmt_mean" in df.columns:
        df["amt_to_mean_card1"] = df["TransactionAmt"] / (df["card1_TransactionAmt_mean"] + 1e-5)

    if {"card4", "ProductCD"}.issubset(df.columns):
        df["card4_ProductCD"] = df["card4"].astype(str) + "_" + df["ProductCD"].astype(str)

    print(f"[INFO] After ratio & interaction features: {df.shape}")
    return df

In [8]:
# =========================================================
# 7️⃣ PCA (V-features)
# =========================================================
def reduce_v_features(df: pd.DataFrame, n_components: int = 30) -> pd.DataFrame:
    v_cols = [c for c in df.columns if c.startswith("V")]
    if v_cols and df[v_cols].var().sum() > 0:
        pca = PCA(n_components=n_components, random_state=42)
        v_pca = pca.fit_transform(df[v_cols].fillna(0))
        v_pca_df = pd.DataFrame(v_pca, columns=[f"V_pca_{i+1}" for i in range(n_components)], index=df.index)
        df = pd.concat([df.drop(columns=v_cols), v_pca_df], axis=1)
        print(f"[INFO] PCA reduced {len(v_cols)} → {n_components}")

        print(f"[INFO] After PCA: {df.shape}")
    return df

In [9]:
# =========================================================
# 8️⃣ SAVE & LOG
# =========================================================
def save_outputs(df: pd.DataFrame, cleaned_path: str, log_path: str):
    print(f"[INFO] Saving cleaned data to {cleaned_path} and log to {log_path}, data shape: {df.shape}")
    df.to_parquet(cleaned_path, index=False)
    log = {
        "rows": len(df),
        "cols": len(df.columns),
        "missing_after_processing": int(df.isna().sum().sum())
    }
    with open(log_path, "w") as f:
        json.dump(log, f, indent=4)
    print(f"[INFO] Saved: {cleaned_path}")

In [10]:
# =========================================================
# 🚀 MAIN PIPELINE
# =========================================================
def build_features(input_path: str, cleaned_path: str, log_path: str):
    df = load_data(input_path)
    df = encode_categorical(df)
    df = transform_numerical(df)
    df = add_temporal_features(df)
    df = add_aggregations(df)
    df = add_ratios_interactions(df)
    df = reduce_v_features(df)
    save_outputs(df, cleaned_path, log_path)
    print("[SUCCESS] Feature engineering pipeline completed.")
    return df

In [11]:
build_features(
        input_path=r"E:\E2E Project 1\E2E-Project-1\fraud-detection-ml\data\parquet\train_data_cleaned.parquet",
        cleaned_path=r"E:\E2E Project 1\E2E-Project-1\fraud-detection-ml\data\parquet\train_cleaned_final.parquet",
        log_path=r"E:\E2E Project 1\E2E-Project-1\fraud-detection-ml\data\imputation_log.json"
    )

[INFO] Loaded data: (590540, 360)
[INFO] Low: 22, Medium: 2, High: 2
Dropped column count in categorical encoding: 26
Newly added column count in categorical encoding: 78
[INFO] After categorical encoding: (590540, 412)
[INFO] After numeric transformations: (590540, 413)
[INFO] New columns added : hour, day, week, is_weekend
[INFO] After temporal features: (590540, 417)
[INFO] New columns added through aggregations  5
[INFO] After aggregations: (590540, 422)
[INFO] After ratio & interaction features: (590540, 423)
[INFO] PCA reduced 292 → 30
[INFO] After PCA: (590540, 161)
[INFO] Saving cleaned data to E:\E2E Project 1\E2E-Project-1\fraud-detection-ml\data\parquet\train_cleaned_final.parquet and log to E:\E2E Project 1\E2E-Project-1\fraud-detection-ml\data\imputation_log.json, data shape: (590540, 161)
[INFO] Saved: E:\E2E Project 1\E2E-Project-1\fraud-detection-ml\data\parquet\train_cleaned_final.parquet
[SUCCESS] Feature engineering pipeline completed.


,TransactionID,isFraud,TransactionDT,TransactionAmt,card1,card2,card3,card5,addr1,addr2,...,V_pca_21,V_pca_22,V_pca_23,V_pca_24,V_pca_25,V_pca_26,V_pca_27,V_pca_28,V_pca_29,V_pca_30
0,2987000,0,86400,68.50,13926,361.0,150.0,142.0,315.0,87.0,...,-9.908050,-12.183862,23.004013,2.855745,0.812381,1.154273,3.069075,3.865351,-0.547806,7.197223
1,2987001,0,86401,29.00,2755,404.0,150.0,102.0,325.0,87.0,...,-3.379099,1.612890,5.837529,-3.140856,1.280215,-2.533151,1.720882,-2.167913,0.093913,1.755993
2,2987002,0,86469,59.00,4663,490.0,150.0,166.0,330.0,87.0,...,-3.379072,1.612903,5.837716,-3.140574,1.280397,-2.533164,1.721346,-2.168462,0.094838,1.755521
3,2987003,0,86499,50.00,18132,567.0,150.0,117.0,476.0,87.0,...,160.420690,-66.586955,116.003944,40.603934,-34.791342,7.591256,-38.829521,2.050958,24.087463,-23.648639
4,2987004,0,86506,50.00,4497,514.0,150.0,102.0,420.0,87.0,...,-3.379593,1.613117,5.837541,-3.140782,1.279934,-2.531991,1.721223,-2.167111,0.094686,1.755773
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3577535,0,15811047,49.00,6550,361.0,150.0,226.0,272.0,87.0,...,-6.456174,-14.638264,-27.439472,-3.747440,4.890997,-4.250772,-7.051789,-5.754606,-13.763901,-11.201304
590536,3577536,0,15811049,39.50,10444,225.0,150.0,224.0,204.0,87.0,...,-3.379072,1.612903,5.837716,-3.140574,1.280397,-2.533164,1.721346,-2.168462,0.094838,1.755521
590537,3577537,0,15811079,30.95,12037,595.0,150.0,224.0,231.0,87.0,...,-3.378910,1.611893,5.836179,-3.140654,1.281054,-2.535113,1.719038,-2.170578,0.093551,1.754911
590538,3577538,0,15811088,117.00,7826,481.0,150.0,224.0,387.0,87.0,...,-78.350018,-258.673725,27.371399,3.636614,-186.239859,-18.354140,-62.693851,-11.014454,36.358398,58.489117
